### Phase 1.3: Spatial Regridding & Export Script

In [2]:
import numpy as np
import rasterio
from rasterio.enums import Resampling
from rasterio.transform import from_bounds
import xarray as xr
import os
import gc
import pandas as pd
import re

# ---------------------------------------------------------
# 1. MASTER GRID & CONSTANTS
# ---------------------------------------------------------
START_YEAR = 2018
END_YEAR = 2025

# Target Bounding Box
LAT_MIN, LAT_MAX = 27.50, 29.20
LON_MIN, LON_MAX = 76.15, 78.40

# Target Grid Dimensions
N_LAT = 141
N_LON = 231

lats = np.linspace(LAT_MAX, LAT_MIN, N_LAT)
lons = np.linspace(LON_MIN, LON_MAX, N_LON)

# ---------------------------------------------------------
# 2. HELPER FUNCTIONS (UPDATED FOR S5P TIME ALIGNMENT)
# ---------------------------------------------------------
def read_and_resize_tif(filepath, expected_days, year):
    """Reads a GEE GeoTIFF, resamples it, and time-aligns missing days."""
    if not os.path.exists(filepath):
        print(f"  [!] Missing file: {filepath}. Filling with NaNs.")
        return np.full((expected_days, N_LAT, N_LON), np.nan, dtype=np.float32)
        
    with rasterio.open(filepath) as src:
        actual_bands = src.count
        
        # Read the exact number of bands that exist in the file
        raw_data = src.read(
            out_shape=(actual_bands, N_LAT, N_LON),
            resampling=Resampling.bilinear
        )
        
        # If the file has all 365/366 days, return immediately
        if actual_bands == expected_days:
            return raw_data
            
        print(f"  [*] Shape mismatch in {os.path.basename(filepath)}: Found {actual_bands}/{expected_days} bands. Auto-aligning...")
        
        # Initialize an empty array filled with NaNs
        aligned_data = np.full((expected_days, N_LAT, N_LON), np.nan, dtype=np.float32)
        
        # Generate expected date strings (YYYY_MM_DD) to match against metadata
        expected_dates = pd.date_range(start=f"{year}-01-01", end=f"{year}-12-31", freq='D').strftime('%Y_%m_%d').tolist()
        
        mapped_count = 0
        for i, desc in enumerate(src.descriptions):
            if desc is None: 
                continue
            # Extract date from description (e.g., "2018_07_10")
            match = re.search(r'(\d{4}_\d{2}_\d{2})', desc)
            if match:
                date_str = match.group(1)
                if date_str in expected_dates:
                    day_idx = expected_dates.index(date_str)
                    aligned_data[day_idx] = raw_data[i]
                    mapped_count += 1
                    
        # Fallback: If GEE stripped the metadata, we know S5P launched mid-2018, 
        # so we pad the missing days at the start of the year.
        if mapped_count == 0:
            missing_days = expected_days - actual_bands
            aligned_data[missing_days:] = raw_data
            print(f"      -> Padded {missing_days} missing days at the start of the year.")
        else:
            print(f"      -> Successfully mapped {mapped_count} bands using metadata.")
            
        return aligned_data

def read_and_regrid_wind(filepath, expected_days):
    """Reads ERA5 NetCDF and interpolates to the target grid."""
    if not os.path.exists(filepath):
        print(f"  [!] Missing file: {filepath}. Filling with NaNs.")
        return (np.full((expected_days, N_LAT, N_LON), np.nan, dtype=np.float32),
                np.full((expected_days, N_LAT, N_LON), np.nan, dtype=np.float32))
    
    ds = xr.open_dataset(filepath)
    ds_interp = ds.interp(latitude=lats, longitude=lons, method="linear")
    
    u_wind = ds_interp['u10'].values.astype(np.float32)
    v_wind = ds_interp['v10'].values.astype(np.float32)
    return u_wind, v_wind

# ---------------------------------------------------------
# 3. EXECUTION: BUILDING THE DATACUBES
# ---------------------------------------------------------
print("Starting Phase 1.3: Aligning & Building Yearly DataCubes...")

for year in range(START_YEAR, END_YEAR + 1):
    print(f"\nProcessing {year}...")
    days_in_year = 366 if year % 4 == 0 else 365
    
    # 1. Load & Regrid GEE Variables (Now passing 'year' for alignment)
    aod    = read_and_resize_tif(f"Delhi_NCR_AOD_{year}.tif", days_in_year, year)
    no2    = read_and_resize_tif(f"Delhi_NCR_NO2_{year}.tif", days_in_year, year)
    co     = read_and_resize_tif(f"Delhi_NCR_CO_{year}.tif", days_in_year, year)
    aer_ai = read_and_resize_tif(f"Delhi_NCR_AER_AI_{year}.tif", days_in_year, year)
    
    # 2. Load & Regrid Copernicus Wind
    wind_u, wind_v = read_and_regrid_wind(f"Delhi_NCR_Wind_{year}.nc", days_in_year)
    
    # 3. Initialize Master 4D Tensor: (Days, Channels, Lat, Lon)
    master_cube = np.zeros((days_in_year, 6, N_LAT, N_LON), dtype=np.float32)
    master_cube[:, 0, :, :] = aod
    master_cube[:, 1, :, :] = no2
    master_cube[:, 2, :, :] = co
    master_cube[:, 3, :, :] = aer_ai
    master_cube[:, 4, :, :] = wind_u
    master_cube[:, 5, :, :] = wind_v
    
    # 4. Flatten for GeoTIFF export
    total_bands = days_in_year * 6
    flattened_cube = master_cube.reshape(total_bands, N_LAT, N_LON)
    
    # 5. Write to Final DataCube GeoTIFF
    output_filename = f"Delhi_NCR_DataCube_{year}.tif"
    transform = from_bounds(LON_MIN, LAT_MIN, LON_MAX, LAT_MAX, N_LON, N_LAT)
    
    with rasterio.open(
        output_filename, 'w', 
        driver='GTiff',
        height=N_LAT, width=N_LON,
        count=total_bands, 
        dtype='float32',
        crs='EPSG:4326', 
        transform=transform,
        compress='lzw'
    ) as dst:
        dst.write(flattened_cube)
        
    print(f"  -> Saved {output_filename} successfully. Shape: {flattened_cube.shape}")
    
    del aod, no2, co, aer_ai, wind_u, wind_v, master_cube, flattened_cube
    gc.collect()

print("\nPhase 1 Complete! Data acquisition and spatial alignment are officially finished.")

Starting Phase 1.3: Aligning & Building Yearly DataCubes...

Processing 2018...
  [*] Shape mismatch in Delhi_NCR_NO2_2018.tif: Found 187/365 bands. Auto-aligning...
      -> Successfully mapped 187 bands using metadata.
  [*] Shape mismatch in Delhi_NCR_CO_2018.tif: Found 181/365 bands. Auto-aligning...
      -> Successfully mapped 181 bands using metadata.
  [*] Shape mismatch in Delhi_NCR_AER_AI_2018.tif: Found 187/365 bands. Auto-aligning...
      -> Successfully mapped 187 bands using metadata.
  -> Saved Delhi_NCR_DataCube_2018.tif successfully. Shape: (2190, 141, 231)

Processing 2019...
  -> Saved Delhi_NCR_DataCube_2019.tif successfully. Shape: (2190, 141, 231)

Processing 2020...
  -> Saved Delhi_NCR_DataCube_2020.tif successfully. Shape: (2196, 141, 231)

Processing 2021...
  -> Saved Delhi_NCR_DataCube_2021.tif successfully. Shape: (2190, 141, 231)

Processing 2022...
  [*] Shape mismatch in Delhi_NCR_NO2_2022.tif: Found 364/365 bands. Auto-aligning...
      -> Successfully